# Bank Term Deposit — Classification & Customer Prioritisation

**Objective:** Predict whether a customer subscribes to a term deposit (`y = 1`) and explore how a bank might prioritise contacts when sales capacity is limited.

**Experiment flow:** inspect the data → prevent leakage → prepare features → Logistic Regression baseline → Random Forest comparison → assess decision thresholds → rank customers by predicted probability.

*Dataset:* UCI Bank Marketing (`bank-additional.csv`, semicolon-separated). Place the CSV beside this notebook before running. The smaller dataset has 4,119 rows.

## 1. Load the data and understand the target

The positive class is `yes` (a subscription). Because relatively few customers subscribe, **accuracy alone is misleading**: a classifier predicting `no` for nearly everyone could still appear accurate.

In [1]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = pd.read_csv("bank-additional.csv", sep=";")
print("Dataset shape:", df.shape)
display(df.head())
print("Target distribution:")
display(df["y"].value_counts(normalize=True).rename("proportion"))

Dataset shape: (4119, 21)


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,30,blue-collar,married,basic.9y,no,yes,no,cellular,may,fri,...,2,999,0,nonexistent,-1.8,92.893,-46.2,1.313,5099.1,no
1,39,services,single,high.school,no,no,no,telephone,may,fri,...,4,999,0,nonexistent,1.1,93.994,-36.4,4.855,5191.0,no
2,25,services,married,high.school,no,yes,no,telephone,jun,wed,...,1,999,0,nonexistent,1.4,94.465,-41.8,4.962,5228.1,no
3,38,services,married,basic.9y,no,unknown,unknown,telephone,jun,fri,...,3,999,0,nonexistent,1.4,94.465,-41.8,4.959,5228.1,no
4,47,admin.,married,university.degree,no,yes,no,cellular,nov,mon,...,1,999,0,nonexistent,-0.1,93.200,-42.0,4.191,5195.8,no


Target distribution:


y
no     0.890507
yes    0.109493
Name: proportion, dtype: float64

## 2. Define features and keep the test set separate

Drop `duration`: the duration of a sales call is only known **after** that call, so using it to decide whom to call would leak future information into the model. Encode `yes` as 1 and `no` as 0.

Use an **80/20 stratified split** so the minority class is represented in both partitions. All exploratory model comparisons below use training data only; the test set is reserved for a final demonstration.

In [2]:
X = df.drop(columns=["y", "duration"])
y = df["y"].map({"no": 0, "yes": 1})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("Training rows:", len(X_train), "| Test rows:", len(X_test))
print("Training subscription rate:", f"{y_train.mean():.2%}")

Training rows: 3295 | Test rows: 824
Training subscription rate: 10.96%


## 3. Explore *training* features

Check data types, missingness, and numeric ranges before choosing preprocessing. Note that `unknown` in a categorical column is a string category, not a missing (`NaN`) value.

In [3]:
display(X_train.head())
print("Feature dtypes:")
print(X_train.dtypes.value_counts())
display(pd.DataFrame({
    "missing": X_train.isna().sum(),
    "missing_%": (100 * X_train.isna().mean()).round(2),
    "unique": X_train.nunique(dropna=True),
}).sort_values("missing", ascending=False))
display(X_train.describe(include="number").T)

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed
2782,48,admin.,divorced,university.degree,no,no,no,cellular,may,tue,3,999,0,nonexistent,-1.8,92.893,-46.2,1.291,5099.1
2547,34,services,unknown,high.school,no,yes,yes,cellular,may,tue,2,999,1,failure,-1.8,92.893,-46.2,1.344,5099.1
2994,47,blue-collar,single,basic.9y,unknown,no,no,cellular,may,wed,1,999,0,nonexistent,-1.8,92.893,-46.2,1.334,5099.1
3626,51,technician,married,professional.course,no,yes,yes,cellular,aug,thu,3,999,0,nonexistent,1.4,93.444,-36.1,4.963,5228.1
2462,29,blue-collar,single,high.school,no,yes,yes,cellular,may,wed,1,999,0,nonexistent,-1.8,92.893,-46.2,1.281,5099.1


Feature dtypes:
object     10
float64     5
int64       4
Name: count, dtype: int64


,missing,missing_%,unique
age,0,0.0,65
job,0,0.0,12
marital,0,0.0,4
education,0,0.0,8
default,0,0.0,3
housing,0,0.0,3
loan,0,0.0,3
contact,0,0.0,2
month,0,0.0,10
day_of_week,0,0.0,5


,count,mean,std,min,25%,50%,75%,max
age,3295.0,40.184825,10.373130,18.000,32.000,38.000,47.000,86.000
campaign,3295.0,2.513505,2.541170,1.000,1.000,2.000,3.000,35.000
pdays,3295.0,958.307739,196.905440,0.000,999.000,999.000,999.000,999.000
previous,3295.0,0.203035,0.558961,0.000,0.000,0.000,0.000,6.000
emp.var.rate,3295.0,0.068437,1.568603,-3.400,-1.800,1.100,1.400,1.400
cons.price.idx,3295.0,93.574734,0.582684,92.201,93.075,93.444,93.994,94.767
cons.conf.idx,3295.0,-40.404825,4.632171,-50.800,-42.700,-41.800,-36.400,-26.900
euribor3m,3295.0,3.608195,1.738085,0.635,1.334,4.857,4.961,5.045
nr.employed,3295.0,5165.577724,74.286875,4963.600,5099.100,5191.000,5228.100,5228.100


## 4. Preprocessing and Logistic Regression baseline

Standardise numerical features (useful for Logistic Regression) and one-hot encode categorical features. Keep preprocessing **inside the pipeline** so every cross-validation fold fits its transformations using only that fold's training rows. Random Forest doesn't need scaling, but reusing the same transformer makes this comparison straightforward.

In [4]:
numeric_cols = X_train.select_dtypes(include="number").columns.tolist()
categorical_cols = X_train.select_dtypes(
    include=["object", "category", "string", "bool"]
).columns.tolist()
assert set(numeric_cols) | set(categorical_cols) == set(X_train.columns)

preprocessor = ColumnTransformer([
    ("numeric", StandardScaler(), numeric_cols),
    ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
])

logistic_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000)),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## 5. Evaluate models on out-of-fold training predictions

`cross_val_predict(..., method="predict_proba")` fits separate models on folds and obtains a prediction for every training row **from a model that did not train on that row**. It does **not** fit the original pipeline in place.

Compare thresholds of 0.5, 0.3 and 0.2:

- **Precision:** Of those flagged to contact, how many actually subscribed?
- **Recall:** Of all subscribers, how many did we identify?
- **F1:** Harmonic mean of precision and recall.

Lowering the threshold usually increases recall while reducing precision. These are exploratory thresholds, not a calibrated optimisation.

In [5]:
def threshold_table(actual, probabilities, thresholds=(0.5, 0.3, 0.2)):
    rows = []
    for threshold in thresholds:
        predictions = (probabilities >= threshold).astype(int)
        precision, recall, f1, _ = precision_recall_fscore_support(
            actual, predictions, average="binary", zero_division=0
        )
        rows.append({
            "threshold": threshold,
            "precision": precision,
            "recall": recall,
            "f1": f1,
        })
    return pd.DataFrame(rows)

logistic_oof_probs = cross_val_predict(
    logistic_pipeline, X_train, y_train, cv=cv, method="predict_proba"
)[:, 1]
logistic_results = threshold_table(y_train, logistic_oof_probs)
print("Logistic Regression — out-of-fold training estimates")
display(logistic_results.round(3))

Logistic Regression — out-of-fold training estimates


,threshold,precision,recall,f1
0,0.5,0.596,0.233,0.335
1,0.3,0.470,0.418,0.443
2,0.2,0.449,0.529,0.486


## 6. Try Random Forest (including class weighting)

A Random Forest can capture non-linear relationships and feature interactions. `class_weight="balanced"` gives the minority class greater weight during training, but **does not guarantee better F1 or precision**. Keep the Logistic Regression results: a more complex model is not automatically better.

In [6]:
forest_pipeline = clone(logistic_pipeline).set_params(
    model=RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    )
)
forest_oof_probs = cross_val_predict(
    forest_pipeline, X_train, y_train, cv=cv, method="predict_proba"
)[:, 1]
forest_results = threshold_table(y_train, forest_oof_probs)
print("Random Forest — out-of-fold training estimates")
display(forest_results.round(3))

comparison = pd.concat([
    logistic_results.assign(model="Logistic Regression"),
    forest_results.assign(model="Random Forest"),
], ignore_index=True)
display(comparison[["model", "threshold", "precision", "recall", "f1"]].round(3))

Random Forest — out-of-fold training estimates


,threshold,precision,recall,f1
0,0.5,0.569,0.205,0.301
1,0.3,0.440,0.424,0.432
2,0.2,0.383,0.551,0.452


,model,threshold,precision,recall,f1
0,Logistic Regression,0.5,0.596,0.233,0.335
1,Logistic Regression,0.3,0.470,0.418,0.443
2,Logistic Regression,0.2,0.449,0.529,0.486
3,Random Forest,0.5,0.569,0.205,0.301
4,Random Forest,0.3,0.440,0.424,0.432
5,Random Forest,0.2,0.383,0.551,0.452


## 7. Business interpretation: a fixed calling budget

The right metric depends on the decision. If the sales team can call **at most 100 customers**, rank customers by predicted subscription probability and examine **Precision@100**:

$$
\mathrm{Precision@100} = \frac{\text{Actual subscribers among top 100}}{100}
$$

Unlike threshold-based precision, this metric measures the yield of a **fixed number of calls**. This is a useful *demonstration* of a contact-ranking strategy; choosing the final model and contact budget rigorously would require comparing ranking metrics on validation folds and accounting for the economics of calls.

For continuity with the exercise, we fit the Random Forest on **all training rows** and inspect its final test-set ranking. This does not establish that it is better than Logistic Regression at ranking.

In [7]:
# cross_val_predict above did not fit forest_pipeline itself.
forest_pipeline.fit(X_train, y_train)
forest_test_probs = forest_pipeline.predict_proba(X_test)[:, 1]

k = min(100, len(X_test))
top_100 = np.argsort(forest_test_probs)[-k:][::-1]  # highest first

top_10 = X_test.iloc[top_100[:10]].copy()
top_10["conversion_prob"] = forest_test_probs[top_100[:10]]
top_10["actual_y"] = y_test.iloc[top_100[:10]].to_numpy()

print("Top 10 ranked test customers:")
display(top_10)
print(f"Precision@10:  {y_test.iloc[top_100[:10]].mean():.2%}")
print(f"Precision@{k}: {y_test.iloc[top_100].mean():.2%}")
print(f"Test-set baseline positive rate: {y_test.mean():.2%}")

Top 10 ranked test customers:


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,conversion_prob,actual_y
2117,31,unemployed,single,university.degree,no,yes,no,cellular,mar,tue,...,10,1,success,-1.8,93.369,-34.8,0.646,5008.7,0.910,1
619,35,services,married,professional.course,no,yes,no,cellular,apr,thu,...,6,1,success,-1.8,93.749,-34.6,0.644,5008.7,0.880,1
2138,33,admin.,single,university.degree,no,no,no,cellular,jun,wed,...,3,1,success,-1.7,94.055,-39.8,0.715,4991.6,0.810,1
2661,47,admin.,married,university.degree,no,no,no,cellular,aug,wed,...,4,2,success,-1.7,94.027,-38.3,0.900,4991.6,0.805,1
673,25,admin.,single,university.degree,no,yes,no,cellular,sep,thu,...,1,1,success,-3.4,92.379,-29.8,0.809,5017.5,0.785,0
977,26,student,single,basic.9y,no,yes,no,cellular,mar,mon,...,999,0,nonexistent,-1.8,92.843,-50.0,1.811,5099.1,0.730,1
3243,45,management,married,unknown,no,yes,no,cellular,apr,tue,...,7,1,success,-1.8,93.749,-34.6,0.642,5008.7,0.720,1
2187,55,admin.,divorced,high.school,no,yes,no,cellular,aug,tue,...,6,3,success,-1.7,94.027,-38.3,0.904,4991.6,0.720,1
2154,30,self-employed,married,university.degree,no,yes,no,cellular,jun,fri,...,999,0,nonexistent,-2.9,92.963,-40.8,1.268,5076.2,0.665,0
1632,29,admin.,married,university.degree,no,no,yes,cellular,jun,tue,...,5,1,success,-1.7,94.055,-39.8,0.702,4991.6,0.655,1


Precision@10:  80.00%
Precision@100: 40.00%
Test-set baseline positive rate: 10.92%


## 8. Threshold-based test evaluation (same fitted forest)

To contrast the ranking use case with binary classification, show the test confusion matrix and precision/recall/F1 at threshold **0.2**, one of the thresholds explored on training folds. This is a *held-out evaluation*, not an invitation to keep changing the threshold based on test results.

In [8]:
chosen_threshold = 0.2
forest_test_preds = (forest_test_probs >= chosen_threshold).astype(int)
print("Confusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y_test, forest_test_preds))
print(classification_report(y_test, forest_test_preds, zero_division=0))

Confusion matrix [[TN, FP], [FN, TP]]:
[[646  88]
 [ 44  46]]
              precision    recall  f1-score   support

           0       0.94      0.88      0.91       734
           1       0.34      0.51      0.41        90

    accuracy                           0.84       824
   macro avg       0.64      0.70      0.66       824
weighted avg       0.87      0.84      0.85       824



## 9. Conclusions and limitations

- **Class imbalance:** Only a small fraction of customers subscribe, so a high overall accuracy can hide weak positive-class recall.
- **The Logistic Regression baseline matters:** Compare its cross-validated threshold metrics with Random Forest before declaring a winner. Class weighting is not a guaranteed improvement.
- **Threshold vs ranking:** F1 helps assess classification, while Precision@100 answers the operational question, *“If we can call 100 customers, how many subscribers are in the selected group?”*
- **Previous-campaign history:** `poutcome="success"` occurs frequently among the top-ranked examples. This is an observation, **not proof of feature importance or causality**.
- **Predicted probability ≠ guaranteed conversion:** Random Forest probabilities may be miscalibrated. Customers with high scores can still decline.
- **Avoid test-set tuning:** Once the held-out results have been viewed, don't use the same test set repeatedly to select algorithms, hyperparameters, thresholds or calling budgets. For a more rigorous follow-up, use nested validation or a new untouched holdout.
- **Dataset constraints:** This historical dataset records subscriptions associated with a marketing campaign. It does not establish that contacting a particular customer *caused* their subscription.

**Exercise takeaway:** A model can have an unimpressive F1 yet still rank leads usefully for a capacity-constrained sales team. Evaluate the model against the *decision it supports*.